# Clinical EHR Analytics: Value-Based Care, Risk Stratification & ROI Simulation
**Author**: Daniel Ademola Dongo  
**Dataset**: Cleaned Clinical EHR Cohort (983 Patients, Zero-Null Validated)  
**Objective**: Quantify clinical risk, uncover therapeutic care gaps, and simulate financial cost avoidance under value-based care benchmarks.


## Section 1: Baseline Clinical Profiling & Exploratory Data Analysis
Establishing demographic, clinical, and pharmacotherapy baselines across the 983-patient EHR cohort.


In [ ]:
# ==============================================================================
# 1.1 Environment Setup & Library Imports
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure all columns and clean widths display in the notebook
pd.set_option('display.max_columns', 35)
pd.set_option('display.width', 1000)

print('Environment and analytical libraries successfully loaded!')


In [ ]:
# ==============================================================================
# 1.2 Data Ingestion & Shape Verification
# ==============================================================================
df = pd.read_csv('../data/raw/cleaned_healthcare_data.csv', parse_dates=['visit_date'])

print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n')


In [ ]:
# ==============================================================================
# 1.3 Dataset Architecture & Schema Inspection
# ==============================================================================
df.info()


In [ ]:
# ==============================================================================
# 1.4 Raw Record Inspection
# ==============================================================================
df.head(5)


In [ ]:
# ==============================================================================
# 1.5 Data Completeness & Missing Value Audit
# ==============================================================================
print('Missing values per column:')
print(df.isna().sum())


### Statistical Profiling of Key Clinical & Numerical Features
Establishing parametric distribution baselines (Mean, SD, IQR) for vascular and demographic vitals.


In [ ]:
# ==============================================================================
# 1.6 Parametric & Biometric Distribution Summary
# ==============================================================================
df[['age', 'systolic_bp', 'diastolic_bp', 'cholesterol']].describe().T


In [ ]:
# ==============================================================================
# 1.7 Categorical Baseline & Condition-Medication Cross-Tabulation
# ==============================================================================
pd.crosstab(df['condition'], df['medication'], margins=True)


In [ ]:
# ==============================================================================
# 1.8 Global Visualization Styling & Clinical Palette
# ==============================================================================
sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.titlesize'] = 12

# Standard Clinical Palette (Defined once globally for cross-visual consistency)
CLINICAL_PALETTE = {
    'Normal': '#10B981',       # Emerald Green
    'Elevated': '#F59E0B',     # Amber
    'Stage 1': '#F97316',      # Orange
    'Stage 2': '#DC2626',      # Crimson
    'Discordant': '#9333EA',   # Purple
    'Neutral': '#64748B'       # Slate Grey
}

print('Global visualization themes and clinical palette initialized!')


## Section 2: Clinical & Operational Feature Engineering
Translating raw biometric vitals and patient contact records into clinical diagnostic stages (AHA/ACC BP guidelines, NCEP cholesterol tiers, composite acuity scores, and contactability status).


In [ ]:
# ==============================================================================
# 2.1 AHA/ACC Blood Pressure Staging & NCEP Cholesterol Classification
# ==============================================================================
def classify_bp(systolic, diastolic):
    """Classifies blood pressure according to AHA/ACC guidelines."""
    if systolic >= 140 and diastolic >= 90:
        return 'Stage 2 Hypertension'
    elif (130 <= systolic <= 139) or (80 <= diastolic <= 89):
        return 'Stage 1 Hypertension'
    elif (120 <= systolic <= 129) and (diastolic < 80):
        return 'Elevated'
    else:
        return 'Normal'

def classify_chol(chol):
    """Classifies total cholesterol according to NCEP ATP III guidelines."""
    if chol >= 240:
        return 'High'
    elif chol >= 200:
        return 'Borderline High'
    else:
        return 'Desirable'

# Apply vector transformations
df['bp_stage'] = [classify_bp(s, d) for s, d in zip(df['systolic_bp'], df['diastolic_bp'])]
df['cholesterol_tier'] = df['cholesterol'].apply(classify_chol)

print('--- Blood Pressure Stage Distribution ---')
print(df['bp_stage'].value_counts())
print('\n--- Cholesterol Tier Distribution ---')
print(df['cholesterol_tier'].value_counts())


In [ ]:
# ==============================================================================
# 2.2 Pharmacotherapy Care Gap & Contraindication Engine
# ==============================================================================
# Canonical clean condition & medication series (DRY - Defined once globally)
cond_clean = df['condition'].astype(str).str.strip().str.lower()
med_clean = df['medication'].astype(str).str.strip().str.lower()

# Chronic conditions requiring disease-modifying pharmacotherapy
chronic_conditions = ['heart disease', 'diabetes', 'hypertension']
df['unmedicated_chronic'] = cond_clean.isin(chronic_conditions) & (med_clean == 'none')

# Therapeutic discordance & clinical contraindications
diabetic_mismatch = (cond_clean == 'diabetes') & (med_clean == 'albuterol')
htn_mismatch = (cond_clean == 'hypertension') & (med_clean == 'albuterol')
asthma_contraindication = (cond_clean == 'asthma') & (med_clean == 'lisinopril')

df['therapeutic_discordance'] = diabetic_mismatch | htn_mismatch | asthma_contraindication

# Categorize care gaps for executive reporting
def categorize_care_gap(row_cond, row_med):
    if row_cond in chronic_conditions and row_med == 'none':
        return 'No Medication Prescribed'
    elif row_cond == 'diabetes' and row_med == 'albuterol':
        return 'Diabetic on Bronchodilator Alone'
    elif row_cond == 'hypertension' and row_med == 'albuterol':
        return 'Hypertensive on Bronchodilator Alone'
    elif row_cond == 'asthma' and row_med == 'lisinopril':
        return 'Asthmatic on ACE-Inhibitor (Contraindicated)'
    elif row_med == 'none':
        return 'Healthy / No Medication Needed'
    else:
        return 'Care Concordant / Under Observation'

df['care_gap_category'] = [categorize_care_gap(c, m) for c, m in zip(cond_clean, med_clean)]
df['care_gap_flagged'] = df['unmedicated_chronic'] | df['therapeutic_discordance']

print('--- Care Gap Summary ---')
print(f"Total Unmedicated Chronic Patients: {df['unmedicated_chronic'].sum()}")
print(f"Total Discordant / Contraindicated Patients: {df['therapeutic_discordance'].sum()}")
print(f"Total Critical Care Gap Cohort: {df['care_gap_flagged'].sum()} ({df['care_gap_flagged'].mean():.1%})\n")
print(df['care_gap_category'].value_counts())


In [ ]:
# ==============================================================================
# 2.3 Operational Outreach & Contact Channel Auditing
# ==============================================================================
email_clean = df['email'].astype(str).str.strip().str.lower()
phone_clean = df['phone_number'].astype(str).str.strip().str.lower()

df['missing_email'] = email_clean.isin(['unknown', 'none', 'nan', ''])
df['missing_phone'] = phone_clean.isin(['unknown', 'none', 'nan', ''])
df['uncontactable'] = df['missing_email'] & df['missing_phone']

def audit_contact_channel(e_miss, p_miss):
    if not e_miss and not p_miss:
        return 'Omnichannel (Email & Phone)'
    elif not e_miss:
        return 'Email Only'
    elif not p_miss:
        return 'Phone Only'
    else:
        return 'Uncontactable (Loss to Follow-up)'

df['contact_status'] = [audit_contact_channel(e, p) for e, p in zip(df['missing_email'], df['missing_phone'])]

print('--- Contact Channel Distribution ---')
print(df['contact_status'].value_counts())
print(f"\nTotal Uncontactable Patients (Loss to Follow-up): {df['uncontactable'].sum()}")


In [ ]:
# ==============================================================================
# 2.4 Evidence-Based Clinical Risk Acuity Scoring & Stratification
# ==============================================================================
def calculate_risk_score(row):
    """
    Computes an evidence-based composite risk score (0 to 100) combining
    vascular aging, AHA BP stage, NCEP cholesterol tier, chronic burden,
    and care gap penalties.
    """
    score = 0

    # 1. Age Factor (Vascular aging)
    age = row['age']
    if age >= 60:
        score += 30
    elif age >= 45:
        score += 20
    elif age >= 35:
        score += 10
    else:
        score += 5

    # 2. Blood Pressure Staging
    bp = row['bp_stage']
    if bp == 'Stage 2 Hypertension':
        score += 35
    elif bp == 'Stage 1 Hypertension':
        score += 20
    elif bp == 'Elevated':
        score += 10

    # 3. Cholesterol Tier
    chol = row['cholesterol_tier']
    if chol == 'High':
        score += 15
    elif chol == 'Borderline High':
        score += 10

    # 4. Chronic Illness Burden
    cond = str(row['condition']).strip().lower()
    if cond == 'heart disease':
        score += 25
    elif cond == 'diabetes':
        score += 20
    elif cond == 'hypertension':
        score += 15
    elif cond == 'asthma':
        score += 10

    # 5. Acute Care Gap Penalty (Unmedicated chronic disease)
    if row['unmedicated_chronic']:
        score += 15

    return min(score, 100)

df['clinical_risk_score'] = df.apply(calculate_risk_score, axis=1)

def map_acuity_tier(score):
    if score >= 75:
        return 'Severe Acuity'
    elif score >= 55:
        return 'High Acuity'
    elif score >= 35:
        return 'Moderate Acuity'
    else:
        return 'Low Acuity'

df['clinical_risk_tier'] = df['clinical_risk_score'].apply(map_acuity_tier)

# High/Severe Acuity uncontactable risk cohort
high_risk_mask = df['clinical_risk_tier'].isin(['High Acuity', 'Severe Acuity'])
df['critical_uncontactable_risk'] = high_risk_mask & df['uncontactable']

print('--- Clinical Risk Acuity Distribution ---')
print(df['clinical_risk_tier'].value_counts())
print(f"\nCritical Uncontactable High/Severe Acuity Patients: {df['critical_uncontactable_risk'].sum()}")


In [ ]:
# ==============================================================================
# 2.5 Feature Engineering Validation & Schema Check
# ==============================================================================
df.head(5)


## Section 3: The 4 Acts of Healthcare Analytics & Business Impact
Translating clinical insights into executive business narratives across 10 strategic questions.


### Act 1: The Clinical Reality — Risk Acuity & Pharmacotherapy Care Gaps
Quantifying baseline clinical vulnerability, unmedicated chronic disease burden, and contraindication risks (Questions 1, 6, 4, 5).


In [ ]:
# ==============================================================================
# Visual 1: Clinical Risk Acuity Matrix (Answers Q1 & Q6)
# ==============================================================================
# 1. Tier Ordering & Palette (Canonical & Global for downstream visuals - DRY)
acuity_order = ['Low Acuity', 'Moderate Acuity', 'High Acuity', 'Severe Acuity']
acuity_colors = {
    'Low Acuity': '#10B981',       # Emerald Green
    'Moderate Acuity': '#F59E0B',  # Gold
    'High Acuity': '#F97316',      # Orange
    'Severe Acuity': '#DC2626'     # Crimson
}

# 2. Initialize Figure and Canvas
fig, ax = plt.subplots(figsize=(12, 7), dpi=100)

# 3. Apply Natural Gaussian Dispersion around Measurement Clusters
np.random.seed(42)
x_jitter = df['cholesterol'] + np.random.normal(0, 2.8, size=len(df))
y_jitter = df['systolic_bp'] + np.random.normal(0, 1.6, size=len(df))

# 4. Scatter Plot using Seaborn
sns.scatterplot(
    x=x_jitter,
    y=y_jitter,
    hue=df['clinical_risk_tier'],
    hue_order=acuity_order,
    palette=acuity_colors,
    alpha=0.65,
    s=75,
    edgecolor='white',
    linewidth=0.4,
    ax=ax
)

# 5. Clinical Reference Lines
ax.axhline(140, color='#DC2626', linestyle='--', linewidth=1.5, label='AHA Stage 2 HTN (≥ 140 mmHg)')
ax.axhline(130, color='#F97316', linestyle=':', linewidth=1.2, label='AHA Stage 1 HTN (130-139 mmHg)')
ax.axvline(200, color='#475569', linestyle='--', linewidth=1.5, label='NCEP High Cholesterol (≥ 200 mg/dL)')

# 6. Highlight Critical Danger Zone (Top-Right Quadrant)
ax.fill_between([195, 230], 135, 147, color='#DC2626', alpha=0.08)
ax.text(202, 142.5, 'CRITICAL DANGER ZONE\n(Combined Stage 2 HTN + High Chol)', color='#991B1B', fontsize=10, weight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEE2E2', edgecolor='#DC2626', alpha=0.85))

# 7. Labels, Limits, and Titles
ax.set_title('Patient Clinical Risk Acuity Matrix\nCross-Tabulating Vitals Against AHA / NCEP Guidelines', fontsize=14, weight='bold', pad=15)
ax.set_xlabel('Serum Cholesterol (mg/dL)', fontsize=12, labelpad=10)
ax.set_ylabel('Systolic Blood Pressure (mmHg)', fontsize=12, labelpad=10)
ax.set_xlim(150, 230)
ax.set_ylim(103, 147)

# 8. Legend Placement
ax.legend(title='Clinical Acuity Tier / Thresholds', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Visual 2: Unmedicated Chronic Care Gap Breakdown (Answers Q4)
# ==============================================================================
# 1. Filter the Unmedicated Chronic Cohort
unmedicated_df = df[df['unmedicated_chronic']].copy()
unmedicated_df['condition_display'] = unmedicated_df['condition'].str.title()

# 2. Category Ordering & Demographic Palette
condition_order = ['Heart Disease', 'Diabetes', 'Hypertension']
age_order = ['<30', '30-44', '60-74']
age_palette = {
    '<30': '#93C5FD',    # Light Blue
    '30-44': '#3B82F6',  # Primary Blue
    '60-74': '#1E3A8A'   # Deep Navy (Senior vulnerable cohort)
}

# 3. Aggregate Data for Bar Plotting
unmed_counts = (
    unmedicated_df.groupby(['condition_display', 'age_group']).size().reset_index(name='patient_count')
)

# 4. Initialize Canvas
fig, ax = plt.subplots(figsize=(11, 6), dpi=100)

# 5. Plot Grouped Horizontal Bars
sns.barplot(
    data=unmed_counts,
    y='condition_display',
    x='patient_count',
    hue='age_group',
    order=condition_order,
    hue_order=age_order,
    palette=age_palette,
    edgecolor='white',
    linewidth=1.2,
    ax=ax
)

# 6. Add Direct Bar Labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=4, fontsize=10, weight='bold', color='#1E293B')

# 7. Executive Callout: Senior Heart Disease Vulnerability
ax.annotate(
    'CRITICAL CARE GAP:\n21 Seniors (60-74) with Heart Disease\nPrescribed ZERO Cardiovascular Therapy',
    xy=(21, 0.28),          # Points to the 60-74 bar on Heart Disease
    xytext=(23, 0.8),        # Callout position
    arrowprops=dict(facecolor='#DC2626', edgecolor='#DC2626', shrink=0.05, width=1.5, headwidth=7),
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#FEF2F2', edgecolor='#DC2626', linewidth=1.5),
    fontsize=9.5,
    fontweight='bold',
    color='#991B1B'
)

# 8. Titles, Labels, Limits, and Legend
ax.set_title('Unmedicated Chronic Disease Cohort by Condition & Demographic\nQuantifying High-Risk Pharmacotherapy Gaps (N = 109)', 
             fontsize=13, weight='bold', pad=15)
ax.set_xlabel('Number of Unmedicated Patients', fontsize=11, labelpad=10)
ax.set_ylabel('Diagnosed Chronic Condition', fontsize=11, labelpad=10)
ax.set_xlim(0, 28)
ax.legend(title='Age Demographic', loc='lower right', frameon=True, framealpha=0.9)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Visual 3: Pharmacotherapy Discordance & Contraindication Matrix (Answers Q5)
# ==============================================================================
import matplotlib.patches as patches

# 1. Prepare Title Case Cross-Tabulation Matrix
matrix_data = pd.crosstab(
    df['condition'].astype(str).str.strip().str.title(),
    df['medication'].astype(str).str.strip().str.title()
)

# Explicit row and column ordering for clinical clarity
row_order = ['Heart Disease', 'Hypertension', 'Diabetes', 'Asthma', 'Unknown']
col_order = ['Lisinopril', 'Metformin', 'Atorvastatin', 'Albuterol', 'None']
matrix_data = matrix_data.reindex(index=row_order, columns=col_order, fill_value=0)

# 2. Initialize Canvas
fig, ax = plt.subplots(figsize=(10, 6.5), dpi=100)

# 3. Render Heatmap
sns.heatmap(
    matrix_data,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar_kws={'label': 'Patient Count'},
    linewidths=1.5,
    linecolor='white',
    annot_kws={'fontsize': 12, 'weight': 'bold'},
    ax=ax
)

# 4. Highlight Specific Clinical Discordance & Contraindications
# Red Bounding Box 1: Asthma on Lisinopril (Row 3, Col 0) -> Absolute Contraindication (ACEI-induced bronchospasm)
rect1 = patches.Rectangle((0, 3), 1, 1, fill=False, edgecolor='#DC2626', linewidth=3.5, linestyle='-')
ax.add_patch(rect1)

# Red Bounding Box 2: Diabetes & HTN on Albuterol Alone (Rows 1 & 2, Col 3) -> Discordant Monotherapy
rect2 = patches.Rectangle((3, 1), 1, 2, fill=False, edgecolor='#F97316', linewidth=3.5, linestyle='--')
ax.add_patch(rect2)

# 5. Executive Callout Notes Below Heatmap
ax.text(
    0.0, -0.22,
    'CRITICAL CLINICAL FINDINGS:\n'
    '• RED BOX (Asthma + Lisinopril, N=41): Absolute Contraindication (ACE-inhibitor induced bronchospasm/cough risk)\n'
    '• ORANGE BOX (HTN/Diabetes + Albuterol, N=70): Therapeutic Discordance (SABA monotherapy with untreated cardio-metabolic disease)',
    transform=ax.transAxes,
    fontsize=9.5, weight='bold', color='#991B1B', va='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#FEF2F2', edgecolor='#DC2626', linewidth=1.2)
)

# 6. Titles and Labels
ax.set_title('Pharmacotherapy Concordance & Prescribing Safety Matrix\nIdentifying Inappropriate Monotherapy and Clinical Contraindications (N = 983)', 
             fontsize=13, weight='bold', pad=15)
ax.set_xlabel('Prescribed Medication Regimen', fontsize=11, labelpad=10)
ax.set_ylabel('Primary Clinical Diagnosis', fontsize=11, labelpad=10)

plt.tight_layout()
plt.show()


### Act 2: The Operational Bottlenecks — Contactability, Surveillance & Data Integrity
Uncovering patient loss-to-follow-up, surveillance recall delays, and clinical data imputation exposure (Questions 2, 3, 8).


In [ ]:
# ==============================================================================
# Visual 4: Contactability Funnel & Loss-to-Follow-up Donut (Answers Q2)
# ==============================================================================
# 1. Define Contact Channel Ordering & Palette (DRY - Defined once globally)
channel_order = ['Omnichannel (Email & Phone)', 'Phone Only', 'Email Only', 'Uncontactable (Loss to Follow-up)']
channel_colors = {
    'Omnichannel (Email & Phone)': '#10B981',  # Emerald Green
    'Phone Only': '#3B82F6',                   # Royal Blue
    'Email Only': '#64748B',                   # Slate Grey
    'Uncontactable (Loss to Follow-up)': '#DC2626' # Crimson
}

channel_counts = df['contact_status'].value_counts().reindex(channel_order)

# 2. Panel 2: Acuity Breakdown of the Uncontactable Cohort (Reusing canonical acuity_order & acuity_colors)
uncontactable_df = df[df['uncontactable']].copy()
# Order tiers Severe -> Low for urgency presentation
uncontactable_acuity = uncontactable_df['clinical_risk_tier'].value_counts().reindex(reversed(acuity_order), fill_value=0)

# Custom helper to show both count and percentage on slices
def make_autopct(values):
    def my_autopct(pct):
        total = sum(values)
        val = int(round(pct * total / 100.0))
        return f'{val}\n({pct:.1f}%)'
    return my_autopct

# 3. Initialize 2-Panel Figure Canvas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=100)

# --- PANEL 1: Contactability Channel Distribution (Horizontal Bar) ---
bars = ax1.barh(channel_counts.index, channel_counts.values, 
                color=[channel_colors[c] for c in channel_counts.index],
                edgecolor='white', linewidth=1.2)
ax1.bar_label(bars, fmt='%d', padding=4, fontsize=10, weight='bold', color='#1E293B')
ax1.set_title('A. Patient Outreach Channel Availability\n145 Patients Completely Uncontactable (N = 983)', 
              fontsize=12, weight='bold', pad=12)
ax1.set_xlabel('Number of Patients', fontsize=11, labelpad=8)
ax1.set_xlim(0, 480)
ax1.invert_yaxis()

# --- PANEL 2: Clinical Acuity of the Uncontactable Cohort (Executive Donut) ---
explode = (0.08, 0.0, 0.0, 0.0)  # Explode Severe Acuity slice

wedges, texts, autotexts = ax2.pie(
    uncontactable_acuity.values,
    labels=uncontactable_acuity.index,
    autopct=make_autopct(uncontactable_acuity.values),
    startangle=140,
    colors=[acuity_colors[a] for a in uncontactable_acuity.index],
    explode=explode,
    pctdistance=0.72,
    textprops={'fontsize': 10, 'weight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(10)
    autotext.set_weight('bold')

# Center Cutout Circle
centre_circle = plt.Circle((0, 0), 0.50, fc='white')
ax2.add_artist(centre_circle)
ax2.text(0, 0.06, f'{len(uncontactable_df)}', ha='center', va='center', fontsize=18, weight='bold', color='#DC2626')
ax2.text(0, -0.12, 'Uncontactable\nPatients', ha='center', va='center', fontsize=9, color='#64748B')

ax2.set_title('B. Clinical Acuity of Uncontactable Cohort\n85 High/Severe Patients Lost to Follow-up', 
              fontsize=12, weight='bold', pad=12)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Visual 5A: Clinical Surveillance Recall Matrix (Answers Q3 - Part 1)
# ==============================================================================
# 1. Feature Engineering: Define Clinical Surveillance Recency Bins
anchor_date = df['visit_date'].max()
df['days_since_visit'] = (anchor_date - df['visit_date']).dt.days

def bin_surveillance(days):
    if days > 365:
        return 'Critically Overdue (>12m)'
    elif days >= 180:
        return 'Recall Due (6-12m)'
    else:
        return 'Active Surveillance (<6m)'

df['surveillance_status'] = df['days_since_visit'].apply(bin_surveillance)
surveillance_order = ['Active Surveillance (<6m)', 'Recall Due (6-12m)', 'Critically Overdue (>12m)']

# 2. Cross-Tabulate (Reusing canonical acuity_order Severe -> Low so row 0 is Severe)
recall_heatmap_data = pd.crosstab(
    df['clinical_risk_tier'],
    df['surveillance_status']
).reindex(index=reversed(acuity_order), columns=surveillance_order)

# 3. Initialize Canvas
fig, ax = plt.subplots(figsize=(10, 5.5), dpi=100)

# 4. Render Heatmap
sns.heatmap(
    recall_heatmap_data,
    annot=True,
    fmt='d',
    cmap='YlOrRd',
    cbar_kws={'label': 'Patient Count'},
    linewidths=1.2,
    linecolor='white',
    annot_kws={'fontsize': 12, 'weight': 'bold'},
    ax=ax
)

# 5. Highlight Top-Right Danger Zone (Severe + High in Critically Overdue: 41 + 79 = 120 pts)
rect_danger = patches.Rectangle((2, 0), 1, 2, fill=False, edgecolor='#991B1B', linewidth=3, linestyle='--')
ax.add_patch(rect_danger)

# 6. Executive Callout Below Matrix
ax.text(
    0.0, -0.20,
    'CRITICAL RECALL CRISIS:\n'
    '• 120 High & Severe Acuity Patients are Critically Overdue (>12 Months Inactive)\n'
    '• Another 114 High/Severe Patients are Due for Recall (6-12 Months Window)',
    transform=ax.transAxes,
    fontsize=9.5, weight='bold', color='#991B1B', va='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#FEF2F2', edgecolor='#DC2626', linewidth=1.2)
)

# 7. Titles and Labels
ax.set_title('Patient Surveillance Matrix: Clinical Risk Acuity vs. Recency of Care\nOperational Prioritization for Patient Recapture (N = 983)', 
             fontsize=13, weight='bold', pad=15)
ax.set_xlabel('Clinical Surveillance Recency Window', fontsize=11, labelpad=10)
ax.set_ylabel('Clinical Acuity Tier', fontsize=11, labelpad=10)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Visual 5B: The Critically Overdue Cohort Deep-Dive (Answers Q3 - Part 2)
# ==============================================================================
# 1. Filter Overdue Cohort (>365 days inactive)
overdue_df = df[df['surveillance_status'] == 'Critically Overdue (>12m)'].copy()

# 2. Disease Breakdown in Overdue Cohort
overdue_conditions = overdue_df['condition'].astype(str).str.strip().str.title().value_counts()

# 3. Channel Breakdown for Overdue Cohort (Reusing channel_order & channel_colors - DRY)
overdue_channels = overdue_df['contact_status'].value_counts().reindex(channel_order, fill_value=0)

# 4. Initialize 2-Panel Figure Canvas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=100)

# --- PANEL 1: Chronic Disease Burden of Overdue Patients (Horizontal Bar) ---
bars1 = ax1.barh(overdue_conditions.index, overdue_conditions.values, 
                 color='#3B82F6', edgecolor='white', linewidth=1.2)
ax1.bar_label(bars1, fmt='%d', padding=4, fontsize=10, weight='bold', color='#1E293B')
ax1.set_title('A. Diagnosed Conditions of Overdue Patients\n53 Heart Disease Patients Inactive >1.8 Years (N = 206)', 
              fontsize=12, weight='bold', pad=12)
ax1.set_xlabel('Number of Patients', fontsize=11, labelpad=8)
ax1.set_xlim(0, 62)
ax1.invert_yaxis()

# Highlight Heart Disease in Crimson
bars1[0].set_color('#DC2626')

# --- PANEL 2: Outreach Feasibility (Executive Donut Chart) ---
explode = (0.0, 0.0, 0.0, 0.08)  # Explode Uncontactable slice

wedges, texts, autotexts = ax2.pie(
    overdue_channels.values,
    labels=overdue_channels.index,
    autopct=make_autopct(overdue_channels.values),
    startangle=140,
    colors=[channel_colors[c] for c in channel_order],
    explode=explode,
    pctdistance=0.72,
    textprops={'fontsize': 10, 'weight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(10)
    autotext.set_weight('bold')

# Center Cutout Circle
centre_circle = plt.Circle((0, 0), 0.50, fc='white')
ax2.add_artist(centre_circle)
ax2.text(0, 0.06, f'{len(overdue_df)}', ha='center', va='center', fontsize=18, weight='bold', color='#DC2626')
ax2.text(0, -0.12, 'Overdue\nPatients', ha='center', va='center', fontsize=9, color='#64748B')

ax2.set_title('B. Outreach Feasibility for Overdue Cohort\n29 Patients Inactive AND Completely Uncontactable', 
              fontsize=12, weight='bold', pad=12)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Visual 6: Data Imputation & Regulatory Audit Provenance (Answers Q8)
# ==============================================================================
# 1. Feature Engineering: Define 4-Tier Data Provenance
chol_imp = df['cholesterol_imputed'].astype(str).str.strip().str.lower() == 'yes'
bp_imp = (df['systolic_bp_imputed'].astype(str).str.strip().str.lower() == 'yes') | \
         (df['diastolic_bp_imputed'].astype(str).str.strip().str.lower() == 'yes')

def audit_provenance(c, b):
    if c and b:
        return 'Both BP & Chol Imputed'
    elif c:
        return 'Cholesterol Only Imputed'
    elif b:
        return 'Blood Pressure Only Imputed'
    else:
        return 'Pure Clinical Measurement'

df['data_provenance'] = [audit_provenance(c, b) for c, b in zip(chol_imp, bp_imp)]
df['has_imputed_vitals'] = chol_imp | bp_imp

prov_order = ['Pure Clinical Measurement', 'Cholesterol Only Imputed', 'Blood Pressure Only Imputed', 'Both BP & Chol Imputed']
prov_colors = {
    'Pure Clinical Measurement': '#10B981',       # Emerald (Audit Compliant)
    'Cholesterol Only Imputed': '#3B82F6',        # Blue
    'Blood Pressure Only Imputed': '#F59E0B',     # Amber
    'Both BP & Chol Imputed': '#DC2626'           # Crimson (High Audit Risk)
}
prov_counts = df['data_provenance'].value_counts().reindex(prov_order)

# 2. Panel 2 Data: Crosstab Imputation x Acuity (Reusing canonical acuity_order - DRY)
acuity_imp = pd.crosstab(
    df['clinical_risk_tier'],
    df['has_imputed_vitals']
).reindex(index=acuity_order)
acuity_imp.columns = ['Clinically Measured', 'Statistically Imputed']

# 3. Initialize 2-Panel Figure Canvas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6.2), dpi=100)

# --- PANEL 1: Data Provenance Distribution (Donut Chart with Pointer) ---
explode_prov = (0.0, 0.0, 0.0, 0.14)

wedges, texts, autotexts = ax1.pie(
    prov_counts.values,
    labels=None,
    autopct=lambda pct: f'{pct:.1f}%' if pct >= 4.0 else '',
    startangle=140,
    colors=[prov_colors[p] for p in prov_order],
    explode=explode_prov,
    radius=1.1,
    pctdistance=0.72,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(11)
    autotext.set_weight('bold')

# Center Cutout Circle
centre_circle = plt.Circle((0, 0), 0.52, fc='white')
ax1.add_artist(centre_circle)
ax1.text(0, 0.06, '35.0%', ha='center', va='center', fontsize=17, weight='bold', color='#DC2626')
ax1.text(0, -0.12, 'Vitals Imputed\n(344 pts)', ha='center', va='center', fontsize=9.5, color='#64748B')

# Dedicated External Pointer Arrow for 2.8% slice
ax1.annotate(
    'Both BP & Chol:\n28 pts (2.8%)',
    xy=(-0.82, 0.82),
    xytext=(-1.55, 1.05),
    arrowprops=dict(facecolor='#DC2626', edgecolor='#DC2626', shrink=0.05, width=1.2, headwidth=5),
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEF2F2', edgecolor='#DC2626', linewidth=1.2),
    fontsize=9.5, weight='bold', color='#991B1B'
)

# Clean Legend positioned below Donut
legend_labels = [f'{k}: {v} ({v/len(df):.1%})' for k, v in prov_counts.items()]
ax1.legend(wedges, legend_labels, loc='lower center', bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=True, fontsize=8.5)
ax1.set_title('A. Clinical Data Provenance Breakdown\nCMS Regulatory Audit Vulnerability (N = 983)', 
              fontsize=12, weight='bold', pad=15)

# --- PANEL 2: Clinical Acuity by Imputation Status (Horizontal Grouped Bar) ---
y_pos = range(len(acuity_imp))
bar_height = 0.38

bars_meas = ax2.barh([p - bar_height/2 for p in y_pos], acuity_imp['Clinically Measured'], 
                     height=bar_height, color='#10B981', label='Clinically Measured', edgecolor='white', linewidth=1.2)
bars_imp = ax2.barh([p + bar_height/2 for p in y_pos], acuity_imp['Statistically Imputed'], 
                    height=bar_height, color='#F97316', label='Statistically Imputed', edgecolor='white', linewidth=1.2)

ax2.bar_label(bars_meas, fmt='%d', padding=4, fontsize=9.5, weight='bold', color='#1E293B')
ax2.bar_label(bars_imp, fmt='%d', padding=4, fontsize=9.5, weight='bold', color='#DC2626')

ax2.set_yticks(y_pos)
ax2.set_yticklabels(acuity_imp.index, fontsize=10, weight='bold')
ax2.set_xlabel('Number of Patients', fontsize=11, labelpad=8)
ax2.set_xlim(0, 265)
ax2.set_title('B. Imputation Exposure by Acuity Tier\n186 High/Severe Patients on Imputed Vitals', 
              fontsize=12, weight='bold', pad=15)
ax2.legend(loc='lower right', frameon=True, framealpha=0.95)

plt.tight_layout()
plt.show()


### Act 3: The Financial Stakes — Inpatient Exposure & Benchmark Penalties
Quantifying acute inpatient admission financial risk and auditing national HEDIS quality metric compliance (Questions 7, 10).


In [ ]:
# ==============================================================================
# Visual 7: Annual Inpatient Cost Exposure by Acuity Tier (Answers Q7)
# ==============================================================================
# 1. Health Economics Model Parameters (AHRQ HCUP Benchmarks)
ADMISSION_COST = 12500  # $12,500 per acute chronic admission

ADMISSION_PROBABILITIES = {
    'Low Acuity': 0.05,
    'Moderate Acuity': 0.15,
    'High Acuity': 0.35,
    'Severe Acuity': 0.65
}

# 2. Compute Projected Admissions and Financial Cost Exposure (Reusing canonical acuity_order & acuity_colors)
tier_counts = df['clinical_risk_tier'].value_counts().reindex(acuity_order)

cost_data = []
for tier in acuity_order:
    count = tier_counts[tier]
    prob = ADMISSION_PROBABILITIES[tier]
    expected_adm = count * prob
    total_cost = expected_adm * ADMISSION_COST
    cost_data.append({
        'clinical_risk_tier': tier,
        'patient_count': count,
        'admission_prob': prob,
        'expected_admissions': expected_adm,
        'cost_exposure': total_cost,
        'cost_millions': total_cost / 1_000_000
    })

cost_df = pd.DataFrame(cost_data)

# High/Severe vs Low/Moderate Pareto Split
high_severe_cost = cost_df[cost_df['clinical_risk_tier'].isin(['High Acuity', 'Severe Acuity'])]['cost_exposure'].sum()
low_mod_cost = cost_df[cost_df['clinical_risk_tier'].isin(['Low Acuity', 'Moderate Acuity'])]['cost_exposure'].sum()
total_exposure = cost_df['cost_exposure'].sum()

# 3. Initialize 2-Panel Figure Canvas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.8), dpi=100)

# --- PANEL 1: Cost Exposure by Acuity Tier (Horizontal Bar) ---
bars1 = ax1.barh(
    cost_df['clinical_risk_tier'],
    cost_df['cost_millions'],
    color=[acuity_colors[t] for t in cost_df['clinical_risk_tier']],
    edgecolor='white',
    linewidth=1.2
)

# Format direct currency labels ($M)
for bar, row in zip(bars1, cost_data):
    width = bar.get_width()
    ax1.text(width + 0.04, bar.get_y() + bar.get_height()/2, 
             f"${width:.2f}M\n({row['expected_admissions']:.1f} adm)", 
             va='center', fontsize=9.5, weight='bold', color='#1E293B')

ax1.set_title('A. Annual Inpatient Financial Exposure by Acuity Tier\nAHRQ HCUP Benchmark ($12,500 / Admission)', 
              fontsize=12, weight='bold', pad=12)
ax1.set_xlabel('Projected Acute Inpatient Exposure ($ Millions)', fontsize=11, labelpad=8)
ax1.set_xlim(0, 2.05)

# --- PANEL 2: Financial Risk Concentration (Pareto Donut) ---
pareto_values = [high_severe_cost, low_mod_cost]
pareto_labels = [
    f'High & Severe Acuity\n${high_severe_cost/1e6:.2f}M (83.2%)',
    f'Low & Moderate Acuity\n${low_mod_cost/1e6:.2f}M (16.8%)'
]
pareto_colors = ['#DC2626', '#3B82F6']

wedges, texts, autotexts = ax2.pie(
    pareto_values,
    labels=pareto_labels,
    autopct='%1.1f%%',
    startangle=140,
    colors=pareto_colors,
    radius=1.1,
    pctdistance=0.72,
    textprops={'fontsize': 10, 'weight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(11)
    autotext.set_weight('bold')

# Center Cutout Circle
centre_circle = plt.Circle((0, 0), 0.52, fc='white')
ax2.add_artist(centre_circle)
ax2.text(0, 0.06, f'${total_exposure/1e6:.2f}M', ha='center', va='center', fontsize=16, weight='bold', color='#0F172A')
ax2.text(0, -0.12, 'Total Annual\nCost Exposure', ha='center', va='center', fontsize=9, color='#64748B')

ax2.set_title('B. 80/20 Pareto Risk Concentration\n83.2% of Total Risk Driven by Top Two Tiers', 
              fontsize=12, weight='bold', pad=12)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Visual 8: HEDIS Quality Benchmark Audit & Compliance Performance (Answers Q10)
# ==============================================================================
# 1. Compute HEDIS Performance (Reusing canonical cond_clean & med_clean - DRY)
htn_mask = cond_clean == 'hypertension'
cbp_rate = ((df.loc[htn_mask, 'systolic_bp'] < 140) & (df.loc[htn_mask, 'diastolic_bp'] < 90)).mean() * 100

hd_mask = cond_clean == 'heart disease'
spc_rate = (med_clean[hd_mask] == 'atorvastatin').mean() * 100

diab_mask = cond_clean == 'diabetes'
supd_rate = (med_clean[diab_mask] == 'atorvastatin').mean() * 100

# 2. Assemble Benchmark Audit Data (Reusing CLINICAL_PALETTE - DRY)
hedis_data = pd.DataFrame([
    {
        'measure_name': f'Controlling Blood Pressure (<140/90)\nHypertensive Cohort (N = {htn_mask.sum()})',
        'actual_rate': cbp_rate,
        'benchmark_rate': 72.0,
        'gap': '+4.5%',
        'status': 'Goal Met',
        'color': CLINICAL_PALETTE['Normal']
    },
    {
        'measure_name': f'Statin Therapy for Cardiovascular Disease\nHeart Disease Cohort (N = {hd_mask.sum()})',
        'actual_rate': spc_rate,
        'benchmark_rate': 80.0,
        'gap': '-53.4%',
        'status': 'Critical Deficit',
        'color': CLINICAL_PALETTE['Stage 2']
    },
    {
        'measure_name': f'Statin Therapy for Patients with Diabetes\nDiabetic Cohort (N = {diab_mask.sum()})',
        'actual_rate': supd_rate,
        'benchmark_rate': 78.0,
        'gap': '-58.4%',
        'status': 'Critical Deficit',
        'color': CLINICAL_PALETTE['Stage 2']
    }
])

# 3. Initialize Figure Canvas
fig, ax = plt.subplots(figsize=(13, 5.5), dpi=100)

y_positions = [0, 1, 2]
bar_height = 0.38

# A. Background full 100% scale track
ax.barh(y_positions, [100, 100, 100], height=bar_height, color='#F1F5F9', edgecolor='none')

# B. Actual Performance Bars
bars = ax.barh(y_positions, hedis_data['actual_rate'], height=bar_height, 
               color=hedis_data['color'], edgecolor='white', linewidth=1.2)

# C. Clean Numbers, Benchmark Ticks, and Right-Hand Status Column
for idx, row in hedis_data.iterrows():
    # Value label: inside if bar is long, outside if bar is short
    val_x = row['actual_rate'] - 2.5 if row['actual_rate'] > 30 else row['actual_rate'] + 2.0
    val_ha = 'right' if row['actual_rate'] > 30 else 'left'
    val_color = 'white' if row['actual_rate'] > 30 else '#1E293B'
    
    ax.text(val_x, idx, f"{row['actual_rate']:.1f}%", 
            va='center', ha=val_ha, fontsize=10.5, weight='bold', color=val_color)
    
    # Target Tick Mark
    ax.plot([row['benchmark_rate'], row['benchmark_rate']], 
            [idx - 0.24, idx + 0.24], 
            color='#0F172A', linewidth=3, solid_capstyle='round')
    
    # Target Percentage Label
    ax.text(row['benchmark_rate'], idx - 0.28, f"Target: {row['benchmark_rate']:.0f}%", 
            ha='center', va='bottom', fontsize=8.5, weight='bold', color='#475569')
    
    # Status Badge with Emoji
    is_met = row['status'] == 'Goal Met'
    badge_bg = '#ECFDF5' if is_met else '#FEF2F2'
    badge_color = '#047857' if is_met else '#991B1B'
    badge_symbol = '✅' if is_met else '⚠️'
    badge_text = f"{badge_symbol} {row['status']} ({row['gap']})"
    
    ax.text(103, idx, badge_text, va='center', ha='left',
            fontsize=9.5, weight='bold', color=badge_color,
            bbox=dict(boxstyle='round,pad=0.4', facecolor=badge_bg, edgecolor=row['color'], linewidth=1.2))

# 4. Styling, Titles, and Limits
ax.set_yticks(y_positions)
ax.set_yticklabels(hedis_data['measure_name'], fontsize=10, weight='bold')
ax.set_xlabel('Compliance / Performance Rate (%)', fontsize=11, labelpad=10)
ax.set_xlim(0, 130)
ax.invert_yaxis()

ax.set_title('National HEDIS Quality Benchmark Audit & Compliance Performance\nExecutive Scorecard: Blood Pressure Control vs. Statin Under-Prescribing', 
             fontsize=12.5, weight='bold', pad=15)

plt.tight_layout()
plt.show()


### Act 4: The Strategic Solution — Preventative Care ROI & Clinical Staffing Model
Proving financial feasibility through multi-scenario preventative care economics and caseload staffing optimization (Questions 9, 9b).


In [ ]:
# ==============================================================================
# Visual 9: Preventative Care Management ROI Simulation (Answers Q9)
# ==============================================================================
# 1. Target High & Severe Acuity Cohort Size (Reusing clinical_risk_tier)
high_severe_mask = df['clinical_risk_tier'].isin(['High Acuity', 'Severe Acuity'])
target_cohort_size = high_severe_mask.sum()  # 553 Patients

# 2. Sensitivity Scenario Data ($ Thousands)
scenarios = ['Conservative', 'Base Case', 'Optimistic']
costs = [276.5, 276.5, 276.5]            # $500/pt annual care management cost
savings = [238.8, 318.4, 477.7]          # Cost avoidance from admission reduction
quality_bonuses = [0.0, 68.0, 136.0]     # CMS 4-Star Quality Incentive Bonus
net_values = [savings[i] + quality_bonuses[i] - costs[i] for i in range(3)]

# 3. Initialize 2-Panel Figure Canvas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.8), dpi=100)

# --- PANEL 1: Scenario Sensitivity Analysis (Grouped Bar Chart) ---
x = np.arange(len(scenarios))
width = 0.22

rects1 = ax1.bar(x - width, costs, width, label='Program Cost ($500/pt)', color='#64748B', edgecolor='white', linewidth=1.2)
rects2 = ax1.bar(x, savings, width, label='Admission Cost Avoidance', color='#3B82F6', edgecolor='white', linewidth=1.2)
rects3 = ax1.bar(x + width, net_values, width, label='Net Economic Value', 
                 color=['#F59E0B' if v < 50 else '#10B981' for v in net_values], edgecolor='white', linewidth=1.2)

ax1.bar_label(rects1, fmt='$%.0fK', padding=3, fontsize=9, weight='bold', color='#1E293B')
ax1.bar_label(rects2, fmt='$%.0fK', padding=3, fontsize=9, weight='bold', color='#1E293B')
ax1.bar_label(rects3, fmt='$%.0fK', padding=3, fontsize=9.5, weight='bold', color='#0F172A')

ax1.set_ylabel('Annual Value ($ Thousands)', fontsize=11, labelpad=8)
ax1.set_title('A. Preventative Care Management Sensitivity\nCost vs. Savings across 3 Operational Scenarios', 
              fontsize=12, weight='bold', pad=12)
ax1.set_xticks(x)
ax1.set_xticklabels(scenarios, fontsize=10.5, weight='bold')
ax1.set_ylim(-50, 560)
ax1.axhline(0, color='#94A3B8', linewidth=1)
ax1.legend(loc='upper left', frameon=True, framealpha=0.95)

# --- PANEL 2: Base Case Net ROI Waterfall Bridge ---
steps = ['Program\nInvestment', 'Inpatient\nSavings', 'Quality\nBonus', 'Net Economic\nValue']
amounts = [-276.5, 318.4, 68.0, 109.9]
step_colors = ['#DC2626', '#3B82F6', '#10B981', '#059669']

bottoms = [0, 0, 318.4, 0]
plot_heights = [276.5, 318.4, 68.0, 109.9]

bars2 = ax2.bar(steps, plot_heights, bottom=bottoms, color=step_colors, width=0.45, edgecolor='white', linewidth=1.2)

ax2.text(0, 276.5/2, '-$276.5K', ha='center', va='center', fontsize=10, weight='bold', color='white')
ax2.text(1, 318.4/2, '+$318.4K', ha='center', va='center', fontsize=10, weight='bold', color='white')
ax2.text(2, 318.4 + 68.0/2, '+$68.0K', ha='center', va='center', fontsize=10, weight='bold', color='white')
ax2.text(3, 109.9/2, '+$109.9K', ha='center', va='center', fontsize=10.5, weight='bold', color='white')

ax2.set_ylabel('Annual Budget Impact ($ Thousands)', fontsize=11, labelpad=8)
ax2.set_title('B. Base Case Net Financial Bridge\n+$109.9K Net Value (140% Program ROI)', 
              fontsize=12, weight='bold', pad=12)
ax2.set_ylim(0, 440)
ax2.axhline(0, color='#94A3B8', linewidth=1)

# Annotate ROI multiplier badge
ax2.text(3, 125, '140% ROI', ha='center', va='bottom', fontsize=10.5, weight='bold', color='#059669',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#ECFDF5', edgecolor='#10B981', linewidth=1.2))

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Visual 10: Care Manager Staffing Optimization & FTE Model (Answers Q9b)
# ==============================================================================
# 1. Staffing Model Parameters (CMSA & NCQA Caseload Standards)
severe_count = (df['clinical_risk_tier'] == 'Severe Acuity').sum()  # 204 pts
high_count = (df['clinical_risk_tier'] == 'High Acuity').sum()      # 349 pts

severe_ratio = 60    # 1 RN per 60 Severe Acuity patients
high_ratio = 100     # 1 RN per 100 High Acuity patients

severe_fte = severe_count / severe_ratio  # 3.40 FTEs
high_fte = high_count / high_ratio        # 3.49 FTEs
total_fte = severe_fte + high_fte         # 6.89 ~ 7.0 FTEs

rn_salary_loaded = 95000  # $95,000 fully loaded annual nurse salary
total_staffing_cost = round(total_fte) * rn_salary_loaded  # 7 * $95K = $665,000

# Annual Value Created: 20% admission reduction + $68K bonus
base_admissions_high_severe = (204 * 0.65) + (349 * 0.35)  # 254.75 admissions
cost_avoidance = base_admissions_high_severe * 0.20 * 12500  # $636,875
annual_value_created = cost_avoidance + (total_fte * 60000)  # Value-based shared savings (~$1,040,000)

# 2. Initialize 2-Panel Figure Canvas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), dpi=100)

# --- PANEL 1: Nurse FTE Allocation by Acuity Tier ---
fte_tiers = ['Severe Acuity\n(1:60 Ratio)', 'High Acuity\n(1:100 Ratio)', 'Total Required\nWorkforce']
fte_values = [severe_fte, high_fte, total_fte]
fte_colors = [acuity_colors['Severe Acuity'], acuity_colors['High Acuity'], '#1E293B']

bars1 = ax1.barh(fte_tiers, fte_values, color=fte_colors, edgecolor='white', linewidth=1.2, height=0.45)

for bar in bars1:
    w = bar.get_width()
    ax1.text(w + 0.15, bar.get_y() + bar.get_height()/2, f'{w:.1f} FTEs', 
             va='center', fontsize=10.5, weight='bold', color='#1E293B')

ax1.set_xlabel('Required Dedicated Nurse Care Managers (FTEs)', fontsize=11, labelpad=8)
ax1.set_xlim(0, 8.5)
ax1.set_title('A. Care Management Caseload Staffing Model\n7.0 Dedicated RN FTEs to Manage 553 High-Risk Patients', 
              fontsize=12, weight='bold', pad=12)
ax1.invert_yaxis()

# --- PANEL 2: Self-Funding Economic Coverage ---
fin_labels = ['Total Nurse\nStaffing Cost', 'Total Economic\nValue Created', 'Net Operating\nMargin']
fin_values = [total_staffing_cost / 1e3, annual_value_created / 1e3, (annual_value_created - total_staffing_cost) / 1e3]
fin_colors = ['#DC2626', '#3B82F6', '#10B981']

bars2 = ax2.bar(fin_labels, fin_values, color=fin_colors, width=0.45, edgecolor='white', linewidth=1.2)

for bar, val in zip(bars2, fin_values):
    y = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, y + 20, f'${val:.0f}K', 
             ha='center', fontsize=10.5, weight='bold', color='#0F172A')

ax2.set_ylabel('Annual Financial Impact ($ Thousands)', fontsize=11, labelpad=8)
ax2.set_ylim(0, 1250)
ax2.set_title('B. Self-Funding ROI Feasibility Model\n$1.04M Value vs. $665K Cost = 156% Self-Funded', 
              fontsize=12, weight='bold', pad=12)

# Self-funding coverage badge
ax2.text(1, 1120, '156% Self-Funded\n(+$375K Net Margin)', ha='center', va='center', 
         fontsize=10, weight='bold', color='#059669',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#ECFDF5', edgecolor='#10B981', linewidth=1.2))

plt.tight_layout()
plt.show()


## Section 4: Production Export & Pipeline Integration
Exporting validated 33-column analytical dataset for executive dashboard deployment and automated unit testing.


In [ ]:
# ==============================================================================
# 4.1 Production Dataset Export & Verification
# ==============================================================================
output_path = '../data/processed/healthcare_analytics_enriched.csv'
df.to_csv(output_path, index=False)

print('=== Production Dataset Successfully Exported ===')
print(f'Location: {output_path}')
print(f'Final Dimensions: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Null Values Remaining: {df.isna().sum().sum()}')
